# 01 — Zillow ZHVI Ingest & Clean

**Purpose.** Download Zillow's county-level ZHVI (Middle Tier (33rd-67th percentile), SFR + Condo/Co-op, Smoothed & Seasonally Adjusted) file, reshape it from Zillow's wide format (one column per month) into a long county-month table, build the county FIPS key, run a coverage audit, and load the result into Postgres as `zhvi_monthly`.


## 1. Setup

In [1]:
import os
import io
import requests
import pandas as pd
from sqlalchemy import text

from housing_risk.config import get_engine

pd.set_option("display.max_columns", 20)


In [2]:
# --- Database connection ---
# Shared with every notebook in this project via src/housing_risk/config.py,
# so credentials and cutoff dates only have to be defined once.
engine = get_engine()

with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())


PostgreSQL 15.18 (Homebrew) on aarch64-apple-darwin25.4.0, compiled by Apple clang version 21.0.0 (clang-2100.0.123.102), 64-bit


## 2. Schema

Creates `zhvi_monthly` if it doesn't already exist. This mirrors `Housing Price Decline/sql/001_schema.sql`'s `zhvi_monthly` table, with `DOUBLE PRECISION` in place of SQLite's `REAL`.

If your team already has a `sql/001_schema.sql` file checked into the repo, run that file directly instead of the inline DDL below (see the commented-out alternative), so the schema stays version-controlled in one place rather than duplicated in this notebook.


In [3]:
DDL = """
CREATE TABLE IF NOT EXISTS zhvi_monthly (
    county_fips   TEXT NOT NULL,
    month         DATE NOT NULL,
    zhvi          DOUBLE PRECISION,
    region_name   TEXT,
    state         TEXT,
    PRIMARY KEY (county_fips, month)
);
"""

with engine.begin() as conn:
    conn.execute(text(DDL))

# Alternative, if sql/001_schema.sql exists in the repo:
# with engine.begin() as conn, open("../sql/001_schema.sql") as f:
#     conn.execute(text(f.read()))


## 3. Download

Zillow ZHVI, Middle Tier (33rd-67th percentile), SFR + Condo/Co-op, Smoothed & Seasonally Adjusted, County geography. This is the same variant used throughout the project (see the Team Reference doc's Data Sources table) — raw/non-seasonally-adjusted is *not* what's loaded here.

Zillow's public CSV endpoints for this file change host paths periodically. If the URL below 404s, grab the current link from Zillow's [Housing Data page](https://www.zillow.com/research/data/) → Home Values → ZHVI → Middle Tier (33rd-67th percentile) → Smoothed, Seasonally Adjusted → County.


In [4]:
ZILLOW_URL = (
    "https://files.zillowstatic.com/research/public_csvs/zhvi/"
    "County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
)

RAW_DIR = "../data/raw/zillow"
os.makedirs(RAW_DIR, exist_ok=True)
raw_path = os.path.join(RAW_DIR, "zhvi_county.csv")

if not os.path.exists(raw_path):
    resp = requests.get(ZILLOW_URL, timeout=60)
    resp.raise_for_status()
    with open(raw_path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content):,} bytes to {raw_path}")
else:
    print(f"Using cached file at {raw_path}")

zhvi_wide = pd.read_csv(raw_path)
zhvi_wide.head()


Using cached file at ../data/raw/zillow/zhvi_county.csv


,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2000-01-31,...,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,210217.606330,...,876483.587025,880007.553689,881688.903831,881664.429746,879853.109780,877155.644127,874790.006512,872663.237476,872220.753950,873008.422497
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,152726.258141,...,327258.568438,328895.003922,330363.857134,332095.627821,333712.155986,335053.111563,336017.428066,337263.418771,339162.650330,341569.750726
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,113021.492070,...,283353.950111,283400.843255,283252.913344,282917.697617,282310.098950,281596.697303,280609.689845,279554.676080,278797.691949,278517.967005
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,146106.503852,...,459582.164053,460367.019987,461220.420505,461894.030995,461740.421962,460735.657745,459147.558173,457589.893930,456733.110469,456252.050797
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,216416.160304,...,925424.340732,927598.249086,928828.819170,929848.898004,930228.329587,930115.693184,929543.266462,928860.113917,929330.517880,930799.654961


## 4. Reshape: wide → long, build county FIPS, trim to each county's coverage window

Zillow ships one column per month (`2000-01-31`, `2000-02-29`, …) alongside identifier columns including `StateCodeFIPS` and `MunicipalCodeFIPS`. Three transforms:

1. **Melt** every date-like column into `month` / `zhvi` rows, one row per county-month, parsing the column headers directly as dates so `month` stays keyed to month-end exactly as Zillow published it.
2. **Concatenate `StateCodeFIPS` + `MunicipalCodeFIPS`** into a single `county_fips` key — Zillow's own columns resolve to county FIPS without needing a separate crosswalk file.
3. **Trim each county to its own first-to-last published month.** Some counties don't have a ZHVI value for the earliest months in the file (Zillow hadn't started covering them yet) or the most recent one or two (not yet published). Those leading/trailing rows are dropped. A NaN that instead falls *inside* a county's active window -- real values both before and after it -- is a genuine gap and is kept as a NULL `zhvi` row rather than silently dropped.

In [5]:
id_cols = [c for c in zhvi_wide.columns if not c[:2].isdigit()]
date_cols = [c for c in zhvi_wide.columns if c not in id_cols]

zhvi_long = zhvi_wide.melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="month",
    value_name="zhvi",
)

# Zillow's column headers are already exact month-end dates (e.g. "2000-01-31",
# "2000-02-29"). Parse them as-is -- routing through .dt.to_period("M").dt.to_timestamp()
# would silently normalize every date to month-*start* instead, which breaks any
# later join keyed on month-end (mortgage_monthly is resampled to month-end too).
zhvi_long["month"] = pd.to_datetime(zhvi_long["month"])

# Concatenate FIPS parts into the county key. Zero-pad: state is 2 digits, county is 3.
zhvi_long["county_fips"] = (
    zhvi_long["StateCodeFIPS"].astype(str).str.zfill(2)
    + zhvi_long["MunicipalCodeFIPS"].astype(str).str.zfill(3)
)

zhvi_clean = zhvi_long.rename(columns={"RegionName": "region_name", "State": "state"})[
    ["county_fips", "month", "zhvi", "region_name", "state"]
]

# Trim each county to its own [first published month, last published month]
# window instead of a blanket dropna(subset=["zhvi"]). A county with no ZHVI
# value before Zillow started covering it (or after it stopped) isn't a data
# gap, it's simply outside that county's reporting window -- dropping those
# rows entirely is correct. A NaN that falls *inside* a county's active window,
# with real values both before and after it, is a genuine gap and stays in the
# table as a NULL zhvi row, the same way a month missing from mortgage_monthly
# surfaces as NULL rather than a dropped row later in the join.
valid = zhvi_clean.dropna(subset=["zhvi"])
coverage_bounds = (
    valid.groupby("county_fips")["month"]
    .agg(first_valid="min", last_valid="max")
)
zhvi_clean = zhvi_clean.merge(coverage_bounds, on="county_fips", how="left")
zhvi_clean = zhvi_clean[
    (zhvi_clean["month"] >= zhvi_clean["first_valid"])
    & (zhvi_clean["month"] <= zhvi_clean["last_valid"])
].drop(columns=["first_valid", "last_valid"])

zhvi_clean.head()

,county_fips,month,zhvi,region_name,state
0,06037,2000-01-31,210217.606330,Los Angeles County,CA
1,17031,2000-01-31,152726.258141,Cook County,IL
2,48201,2000-01-31,113021.492070,Harris County,TX
3,04013,2000-01-31,146106.503852,Maricopa County,AZ
4,06073,2000-01-31,216416.160304,San Diego County,CA


## 5. Coverage audit
Confirm each row has the minimum 2 quarters (6 months) of prior history needed for the 6-month momentum lag feature, and separately flag counties with little or no presence in the training period (training ends 2021-09-30). This doesn't drop rows — it's a diagnostic so Phase 4's feature engineering and Phase 2's split both know what they're working with.


In [6]:
county_month_counts = (
    zhvi_clean.groupby("county_fips")["month"]
    .agg(["min", "max", "count"])
    .rename(columns={"min": "first_month", "max": "last_month", "count": "n_months"})
)

n_counties = county_month_counts.shape[0]
n_short_history = (county_month_counts["n_months"] < 6).sum()

train_cutoff = pd.Timestamp("2021-09-30")
training_presence = (
    zhvi_clean[zhvi_clean["month"] <= train_cutoff]
    .groupby("county_fips")["month"]
    .count()
)
n_no_training_presence = n_counties - training_presence.shape[0]

print(f"Counties: {n_counties:,}")
print(f"Total county-month rows: {len(zhvi_clean):,}")
print(f"Date range: {zhvi_clean['month'].min().date()} to {zhvi_clean['month'].max().date()}")
print(f"Counties with < 6 months of history (can't compute 6-mo momentum yet): {n_short_history:,}")
print(f"Counties with zero rows in the training period (<= {train_cutoff.date()}): {n_no_training_presence:,}")


Counties: 3,071
Total county-month rows: 709,996
Date range: 2000-01-31 to 2026-08-31
Counties with < 6 months of history (can't compute 6-mo momentum yet): 0
Counties with zero rows in the training period (<= 2021-09-30): 50


## 6. Load to Postgres

In [7]:
# Clear existing rows first so this cell is safe to re-run (e.g. after
# changing the cleaning logic, or just re-running the notebook top to bottom)
# -- to_sql(if_exists="append") alone raises a UniqueViolation on
# (county_fips, month) otherwise.
with engine.begin() as conn:
    conn.execute(text("DELETE FROM zhvi_monthly;"))

zhvi_clean.to_sql(
    "zhvi_monthly",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000,
)

with engine.connect() as conn:
    loaded = conn.execute(text("SELECT COUNT(*) FROM zhvi_monthly;")).scalar()
print(f"zhvi_monthly now has {loaded:,} rows")

zhvi_monthly now has 709,996 rows


## 7. Sanity checks

Row counts and a quick look confirm the load matches what was cleaned above, before `02_fred_ingest_clean.ipynb` and the join in `03_join_sql_load.ipynb` build on top of this table.


In [8]:
assert loaded == len(zhvi_clean), "Row count in Postgres doesn't match the cleaned frame — check for a partial or duplicate load."

with engine.connect() as conn:
    sample = pd.read_sql(
        "SELECT * FROM zhvi_monthly ORDER BY county_fips, month LIMIT 5;", conn
    )
sample


,county_fips,month,zhvi,region_name,state
0,01001,2000-01-31 00:00:00,123634.685391,Autauga County,AL
1,01001,2000-02-29 00:00:00,123665.582501,Autauga County,AL
2,01001,2000-03-31 00:00:00,123462.740267,Autauga County,AL
3,01001,2000-04-30 00:00:00,123385.784631,Autauga County,AL
4,01001,2000-05-31 00:00:00,123383.406204,Autauga County,AL
